In [35]:
import pandas as pd
import os, sys, re
import numpy as np
import torch.nn as nn
from sklearn.feature_selection import SelectKBest, f_regression
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import random
import copy
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler


# Import Dataset and Tidy

In [36]:
train_data = pd.read_csv("./data/covid.train.csv")
test_data = pd.read_csv("./data/covid.test.csv")

In [37]:
test_data

,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,shop.2,restaurant.2,spent_time.2,large_event.2,public_transit.2,anxious.2,depressed.2,felt_isolated.2,worried_become_ill.2,worried_finances.2
0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,52.071090,8.624001,29.374792,5.391413,2.754804,19.695098,13.685645,24.747837,66.194950,44.873473
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,58.742461,21.720187,41.375784,9.450179,3.150088,22.075715,17.302077,23.559622,57.015009,38.372829
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,59.109045,20.123959,40.072556,8.781522,2.888209,23.920870,18.342506,24.993341,55.291498,38.907257
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,55.442267,16.083529,36.977612,5.199286,2.575347,21.073800,12.087171,18.608723,67.036197,43.142779
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,60.588783,19.503010,42.631236,11.549771,8.530551,15.896575,11.781634,15.065228,61.196518,43.574676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
888,888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,56.762931,21.494159,44.202567,14.996865,2.291745,17.740003,12.822676,18.123344,60.417531,37.156229
889,889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,57.888461,16.770893,37.373472,7.169675,2.631595,20.587449,15.960166,23.710310,58.758735,38.673787
890,890,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,57.589848,16.761311,36.874822,11.046907,1.912310,16.800220,13.280423,22.423640,60.934851,43.122513
891,891,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,57.966384,22.696669,45.350415,20.343487,2.385330,16.528265,15.092539,17.476063,54.862386,44.016255


In [38]:
train_data

,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,restaurant.2,spent_time.2,large_event.2,public_transit.2,anxious.2,depressed.2,felt_isolated.2,worried_become_ill.2,worried_finances.2,tested_positive.2
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.812411,43.430423,16.151527,1.602635,15.409449,12.088688,16.702086,53.991549,43.604229,20.704935
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.682974,43.196313,16.123386,1.641863,15.230063,11.809047,16.506973,54.185521,42.665766,21.292911
2,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.593983,43.362200,16.159971,1.677523,15.717207,12.355918,16.273294,53.637069,42.972417,21.166656
3,3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.576992,42.954574,15.544373,1.578030,15.295650,12.218123,16.045504,52.446223,42.907472,19.896607
4,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.091433,43.290957,15.214655,1.641667,14.778802,12.417256,16.134238,52.560315,43.321985,20.178428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,2695,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,15.090116,30.839219,7.849525,1.760094,14.617563,11.163213,18.742673,68.024690,38.920206,13.008853
2696,2696,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.779264,30.617100,7.754800,1.780730,14.513419,11.281241,18.539741,67.855755,39.224244,12.725638
2697,2697,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.961085,30.595194,7.744075,1.921828,14.160990,11.163526,18.702564,67.731162,38.740651,12.613441
2698,2698,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.609582,30.420998,7.687974,1.992580,14.409427,11.330301,19.134697,67.795100,38.595125,12.477227


## Baseline 0
先用 sklearn 快速確認「資料整理 + 評估流程」都正確。

In [39]:
# 從 training dataset 中分離出 validation dataset
X = train_data.drop(columns=['id', 'tested_positive.2'])
y = train_data['tested_positive.2'] # day3 percentage
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 標準化
scaler = StandardScaler() # 建立一個「標準化器」
X_train_s = scaler.fit_transform(X_train)

# 沿用上方平均數與標準差來校正 val
X_val_s = scaler.transform(X_val)


# ------------------------------------
# baseline 0 (如果我們預測出來 day3 = day2，loss 會是多少？)
pred_b0 = X_val["tested_positive.1"].to_numpy(dtype=np.float64) 
loss_b0 = mean_squared_error(y_val, pred_b0)
print(f"MSE of baseline 0 (predict y3=y2): {loss_b0}")

MSE of baseline 0 (predict y3=y2): 1.0638112270372913


# Baseline 1
建立 Ridge MODEL（y = wx+b），alpha 是懲罰 ｗ 機制，讓 w 不要太大 -> alpha 越小，model 越接近線性模型，擬合更貼但更可能 overfit；alpha 越大，model 越保守較不容易 overfit


In [40]:
# 測試不同 alpha 的 model

best_alpha_loss = {"alpha":0, "loss":10}

for a in list(np.arange(0.01, 2.0001, 0.01)):
    model = Ridge(alpha=a)
    model.fit(X_train_s, y_train)
    pred = model.predict(X_val_s)
    loss = mean_squared_error(y_val, pred)
    if loss < best_alpha_loss["loss"] and loss < loss_b0:
        best_alpha_loss["alpha"] = a
        best_alpha_loss["loss"] = loss

print(f"best alpha: {best_alpha_loss['alpha']}, MSE of baseline 1: {best_alpha_loss['loss']}")

best alpha: 0.52, MSE of baseline 1: 0.9298377455352191


# PyTorch

### helper functions

In [41]:
def set_seed(seed: int, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# 把已經準備好的訓練資料，轉成「可洗牌、可分批、且在同 seed 下可重現」的訓練迭代器
def make_loader(Xtr, ytr, batch_size: int, seed: int):
    g = torch.Generator()
    g.manual_seed(seed)

    dataset = TensorDataset(Xtr, ytr)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=g,
        num_workers=0
    )

# 新增 features
def add_delta(df):
    df = df.copy()
    # 1. 基礎 Delta 特徵 (原本有的)

    if 'cli.1' in df.columns and 'cli' in df.columns:
        df['cli_delta'] = df['cli.1'] - df['cli']
    if 'ili.1' in df.columns and 'ili' in df.columns:
        df['ili_delta'] = df['ili.1'] - df['ili']
    if 'worried_become_ill.1' in df.columns and 'worried_become_ill' in df.columns:
        df['worried_become_ill_delta'] = df['worried_become_ill.1'] - df['worried_become_ill']
    
    # 2. 【新增】特徵交互作用項 (Interaction Items)
    # 症狀與規模的乘積
    if 'cli.1' in df.columns and 'tested_positive.1' in df.columns:
        df['cli_pos_inter'] = df['cli.1'] * df['tested_positive.1']
    
    # 現狀與趨勢的共振
    if 'cli.1' in df.columns and 'cli_delta' in df.columns:
        df['cli_trend_inter'] = df['cli.1'] * df['cli_delta']
    
    # 指標一致性
    if 'cli.1' in df.columns and 'ili.1' in df.columns:
        df['symptom_inter'] = df['cli.1'] * df['ili.1']
        
    return df


class MultiTaskMLP(nn.Module):
    def __init__(self, input_dim, hidden=64, dropout=0.0, n_layers=1, residual=False):
        super().__init__()
        # Shared Trunk (共享層)
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        # 根據層數堆疊
        trunk_layers = []
        for _ in range(n_layers - 1):
            if residual: 
                trunk_layers.append(ResidualBlock(hidden, dropout))
            else: 
                trunk_layers.append(nn.Sequential(
                                        nn.Linear(hidden, hidden), 
                                        nn.BatchNorm1d(hidden), 
                                        nn.GELU(), 
                                        nn.Dropout(dropout)
                                        )
                                    )
        self.deep_trunk = nn.Sequential(*trunk_layers)

        # Head 1: Main Task (Day 3)
        self.head_main = nn.Sequential(
            nn.Linear(hidden, 1)
        )
        

    def forward(self, x):
        feat = self.trunk(x)
        feat = self.deep_trunk(feat)
        return self.head_main(feat)

# 改為預測 day3-day2 的 delta

In [42]:
X = train_data.drop(columns=['id', 'tested_positive.2'])
y_day3 = train_data['tested_positive.2']
y = y_day3 - train_data['tested_positive.1'] # day3-day2 的 delta
X_train, X_val, y_train, y_val, y_train_day3, y_val_day3 = train_test_split(X, y, y_day3, test_size=0.2, random_state=42)
base_train_raw = X_train['tested_positive.1'].to_numpy(dtype=np.float32)
base_val_raw = X_val['tested_positive.1'].to_numpy(dtype=np.float32)

## Add features

In [43]:
X_train = add_delta(X_train)
X_val = add_delta(X_val)

## Normalization

In [44]:
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train)
Xva_s = scaler.transform(X_val)

## Feature selection (k=16)

In [45]:
k = 32 # 保留的非 baseline features 數量

base_col = X_train.columns.get_loc('tested_positive.1')
base_tr_scaled = Xtr_s[:, base_col:base_col + 1]
base_va_scaled = Xva_s[:, base_col:base_col + 1]
Xtr_no_base = np.delete(Xtr_s, base_col, axis=1)
Xva_no_base = np.delete(Xva_s, base_col, axis=1)

k_eff = min(int(k), Xtr_no_base.shape[1])
selector = SelectKBest(score_func=f_regression, k=k_eff)
Xtr_sel = selector.fit_transform(Xtr_no_base, y_train) # 看特徵 X 跟目標 y 的關係來打分數
Xva_sel = selector.transform(Xva_no_base) # 只保留 fit 時被選中的那些欄位

idx_sel = selector.get_support(indices=True)
selected_sel_cols = X_train.columns.delete(base_col)[idx_sel].tolist()
selected_sel_cols.append('tested_positive.1')
Xtr_s = np.hstack([Xtr_sel, base_tr_scaled])
Xva_s = np.hstack([Xva_sel, base_va_scaled])
selected_sel_cols

['id',
 'ID',
 'IA',
 'wearing_mask',
 'shop',
 'restaurant',
 'spent_time',
 'large_event',
 'worried_become_ill',
 'worried_finances',
 'tested_positive',
 'wearing_mask.1',
 'work_outside_home.1',
 'shop.1',
 'restaurant.1',
 'spent_time.1',
 'large_event.1',
 'worried_become_ill.1',
 'worried_finances.1',
 'cli.2',
 'ili.2',
 'wearing_mask.2',
 'work_outside_home.2',
 'shop.2',
 'restaurant.2',
 'spent_time.2',
 'large_event.2',
 'anxious.2',
 'worried_become_ill.2',
 'worried_finances.2',
 'cli_delta',
 'cli_trend_inter',
 'tested_positive.1']

## 轉換 Tensor 並傳遞

In [ ]:
## ----------- Train Data -----------
Xtr = torch.tensor(Xtr_s, dtype=torch.float32)
ytr = torch.tensor(y_train.to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1)
ytr_day3 = torch.tensor(y_train_day3.to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1)
base_tr = torch.tensor(base_train_raw, dtype=torch.float32).view(-1, 1)

## ----------- Validation Data -----------
Xva = torch.tensor(Xva_s, dtype=torch.float32)
yva = torch.tensor(y_val.to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1)
yva_day3 = torch.tensor(y_val_day3.to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1)
base_va = torch.tensor(base_val_raw, dtype=torch.float32).view(-1, 1)


## ----------- model ----------- 
### initial parameters
seed = 2026
set_seed(seed)
model = MultiTaskMLP(input_dim = Xtr.shape[1],
                     hidden = 64,
                     dropout = 0.2,
                     n_layers = 2,
                     residual = False
        )
lr = 1e-3
bs = 20
epochs = 10000
patience = 10
bad_count = 0
alpha = 0.05
train_hist, val_hist = [], []
best_epoch, best_state = None, None
best_va = float("inf")

### training
opt = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr/100)
train_loader = make_loader(Xtr, ytr, batch_size=bs, seed=seed)

for ep in range(epochs):
    model.train()
    
    for batch in train_loader:
        
        xb, yb = batch
        
        # Input Noise
        noise = torch.randn_like(xb) * 0.005

        pred = model(xb + noise)

        mse = F.mse_loss(pred, yb)
        l2 = sum(p.pow(2).sum() for name, p in model.named_parameters() if not name.endswith("bias"))
        loss =  mse + alpha * l2

        opt.zero_grad()
        loss.backward()
        opt.step()
    
    scheduler.step()

    with torch.no_grad():
        model.eval()

        p_tr_delta = model(Xtr)
        p_va_delta = model(Xva)

        p_tr = p_tr_delta + base_tr
        p_va = p_va_delta + base_va

        tr_mse = F.mse_loss(p_tr, ytr_day3).item()
        va_mse = F.mse_loss(p_va, yva_day3).item()

        train_hist.append(tr_mse)
        val_hist.append(va_mse)
        print(f"epoch {ep + 1}: tr_mse={tr_mse:.4f}, va_mse={va_mse:.4f}")

        if va_mse < best_va - 1e-8:
            best_va = va_mse
            best_epoch = ep + 1
            best_state = copy.deepcopy(model.state_dict())
            bad_count = 0
        else:
            bad_count += 1
            if bad_count >= patience:
                print(f"Early stopping at epoch {ep + 1}. Best epoch = {best_epoch}, best va_mse = {best_va:.4f}")
                break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best model from epoch {best_epoch} with best va_mse = {best_va:.4f}")



## Epoch Curve
訓練完成後，畫出每個 epoch 的 day3 train / validation MSE。


In [ ]:
sns.set_theme(style="whitegrid")
epochs_ran = np.arange(1, len(train_hist) + 1)
best_epoch = int(np.argmin(val_hist) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, train_hist, label="Train MSE", linewidth=2)
plt.plot(epochs_ran, val_hist, label="Validation MSE", linewidth=2)
plt.axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.35, label=f"Best epoch = {best_epoch}")
plt.scatter(best_epoch, val_hist[best_epoch - 1], color="tab:red", s=35, zorder=3)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training / Validation Day3 MSE vs. Epoch")
plt.legend()
plt.tight_layout()
plt.show()
